# Copy Number Heatmap - scDNA (SC) — UMAP / Leiden + MAD Outlier Removal

Loads single-cell DNA copy number data  (primary + brain met), filters gap regions,
then clusters **all cells jointly** via UMAP and Leiden.
Within each cluster, cells too far from the cluster median LogR profile (by Manhattan distance)
are dropped. Remaining cells are ordered by cluster similarity and within-cluster Ward linkage
for the final heatmap.

Two row annotations are shown on every heatmap: **Leiden cluster** and **Sample origin** (Primary / Met).

- `_SC_1_` cells → **Primary**
- `_SC_2_` cells → **Met** (brain metastasis)

UMAP is run directly on LogR values with `metric='manhattan'` and `min_dist=0.1`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import warnings

import sys, os
import importlib, cnvis as cn
importlib.reload(cn)
import cnvis as cn
from cnvis.utilities import load_gaps, genome_range

from scipy.cluster.hierarchy import linkage, leaves_list, to_tree
from sklearn.decomposition import PCA
import umap
import scanpy as sc
import anndata as ad

warnings.simplefilter(action='ignore', category=FutureWarning)
plt.rcParams['figure.dpi'] = 120

## Helper Functions

In [ ]:
SAMPLE_COLORS = {'Primary': '#4A90D9', 'Met': '#E07B54'}

# Shared helpers — see _12_scDNA_helpers.py
from _12_scDNA_helpers import (
    filter_small_clusters, filter_mad_outliers, compute_heatmap_order,
    print_tree, flip_subtree,
    plot_umap,
    cluster_mean_smooth, knn_smooth,
    GENE_CACHE_FILE, GENE_SPAN_PADDING,
)


## Data Preparation

### Load Data

In [ ]:
data_dir = "data2"
data_file = f"{data_dir}/uber.SC_cnv_1.seg.txt"

# Format: chrom, chrompos, abspos, <cell columns...>
cna_data = pd.read_csv(data_file, sep="\t")

cell_names_all = cna_data.columns[3:].tolist()

# Label cells by sample origin
primary_cells = [c for c in cell_names_all if '_SC_1_' in c]
met_cells     = [c for c in cell_names_all if '_SC_2_' in c]
neither_cells = [c for c in cell_names_all if '_SC_1_' not in c and '_SC_2_' not in c]

print(f"Data shape: {cna_data.shape}")
print(f"Number of genomic bins: {len(cna_data)}")
print(f"Total cells: {len(cell_names_all)}")
print(f"  Primary (_SC_1_): {len(primary_cells)}")
print(f"  Met     (_SC_2_): {len(met_cells)}")
print(f"  Neither:             {len(neither_cells)}")
if neither_cells:
    print("  Cells with neither label:", neither_cells)

# sample_origin Series for annotation — built once, filtered downstream as needed
sample_origin_all = pd.Series(
    ['Primary' if '_SC_1_' in c else 'Met' for c in cell_names_all],
    index=cell_names_all, name='Sample'
)

print(f"\nValue range: {cna_data[cell_names_all].min().min():.3f} to {cna_data[cell_names_all].max().max():.3f}")
cna_data.head()

### Preprocess Matrix

Rename `chrompos` → `start`, add `chr` prefix, derive `end` from next bin's start, drop `abspos`.
Then convert ratio → CN (median-normalized) → LogR. The CN intermediate is dropped — `logr_matrix` is the canonical analysis matrix from here on. CN is regenerated locally only in cells that need to render CN tracks.


In [ ]:
# Rename columns to standard format
matrix = cna_data.rename(columns={"chrompos": "start"})

# Add "chr" prefix; remap chromosome 23 → X
matrix["chrom"] = "chr" + matrix["chrom"].astype(int).astype(str)
matrix.loc[matrix["chrom"] == "chr23", "chrom"] = "chrX"

# Drop abspos (not needed)
matrix = matrix.drop(columns=["abspos"])

# Add end column: next bin's start per chromosome.
# sort=False preserves original abspos order (genomic: chr1, chr2, ..., chr22, chrX).
matrix["end"] = (
    matrix.groupby("chrom", sort=False)["start"]
    .shift(-1)
    .fillna(matrix["start"] + 1)
    .astype(int)
)

# Reorder columns: chrom, start, end, then cell columns
matrix = matrix[["chrom", "start", "end"] + cell_names_all]

# ratio → CN (median-normalized per cell) → LogR. CN is intermediate only.
_cn = cn.to_copy_number(matrix, input_type="ratio", ploidy=2)
logr_matrix = pd.concat(
    [_cn[["chrom", "start", "end"]],
     np.log2(_cn[cell_names_all].replace(0, 0.01) / 2).fillna(0)],
    axis=1,
)
del matrix, _cn

print(f"LogR matrix shape (bins x cells): {logr_matrix.shape}")
print(f"LogR value range: {logr_matrix[cell_names_all].min().min():.2f} to {logr_matrix[cell_names_all].max().max():.2f}")
logr_matrix.head()


### Filter Gap Regions

In [ ]:
gap = load_gaps('hg38', include_giab=True)
print(f"Loaded {len(gap)} gap regions for filtering")

logr_matrix = cn.filter_gaps(logr_matrix, gap, method='remove', buffer=0)
print(f"After gap filtering: {logr_matrix.shape}")


## PCA


In [ ]:
pca = PCA(n_components=20, random_state=42)
pca_coords = pca.fit_transform(logr_matrix[cell_names_all].T.values)
explained = pca.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Colour by sample origin in PCA
sample_colors_pca = [SAMPLE_COLORS[sample_origin_all[c]] for c in cell_names_all]
axes[0].scatter(pca_coords[:, 0], pca_coords[:, 1], s=3, alpha=0.4, c=sample_colors_pca)
axes[0].set_xlabel(f'PC1 ({explained[0]:.1f}%)')
axes[0].set_ylabel(f'PC2 ({explained[1]:.1f}%)')
axes[0].set_title('PCA of cells (LogR)')
axes[0].legend(
    handles=[
        Line2D([0], [0], marker='o', color='w', markerfacecolor=col, markersize=6,
               label=f'{lbl} (n={sum(sample_origin_all == lbl)})')
        for lbl, col in SAMPLE_COLORS.items()
    ], fontsize=8
)

axes[1].bar(range(1, len(explained) + 1), explained, color='steelblue')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Explained Variance (%)')
axes[1].set_title('Scree Plot')
axes[1].set_xticks(range(1, len(explained) + 1))
plt.suptitle('PCA Exploration — SC scDNA (Primary + Met)', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Top 5 PCs explain {explained[:5].sum():.1f}% of variance")


## UMAP & Heatmap (Initial)

Cluster all cells via UMAP/Leiden, then plot UMAP (by cluster and by sample) and the initial heatmap (no filtering) — establishes baseline structure before MAD outlier removal.


In [ ]:
n_neighbors = min(30, len(cell_names_all) - 1)

reducer = umap.UMAP(n_components=2, random_state=42,
                    metric='manhattan', min_dist=0.1, n_neighbors=n_neighbors)
umap_coords_all = pd.DataFrame(
    reducer.fit_transform(logr_matrix[cell_names_all].T.values),
    index=cell_names_all, columns=['UMAP1', 'UMAP2'],
)

# Leiden on 2D UMAP coords
resolution = 0.05
adata = ad.AnnData(X=logr_matrix[cell_names_all].T.values)
adata.obsm['X_umap'] = umap_coords_all.values
sc.pp.neighbors(adata, use_rep='X_umap', n_neighbors=n_neighbors)
sc.tl.leiden(adata, resolution=resolution, random_state=42)

leiden_labels_all = adata.obs['leiden'].astype(int).values
n_clusters_all = leiden_labels_all.max() + 1
cluster_series_all = pd.Series(leiden_labels_all, index=cell_names_all, name='cluster')

print(f'Leiden: {n_clusters_all} clusters (resolution={resolution}), {len(cell_names_all)} total cells')
print('\nCluster sizes:')
print(cluster_series_all.value_counts().sort_index().rename(lambda i: f'C{i+1}'))

cluster_labels_all = cluster_series_all.map(lambda i: f'C{i+1}')
plot_umap(umap_coords_all, cluster_labels_all,
          title=f'UMAP + Leiden ({n_clusters_all} clusters) — SC scDNA (Primary + Met)')
plot_umap(umap_coords_all, sample_origin_all, SAMPLE_COLORS,
          title='UMAP by sample origin — SC scDNA (Primary + Met)')

# Initial heatmap — all cells, no filtering, displayed in LogR
sorted_cells_all, cluster_link_all = compute_heatmap_order(cluster_series_all, logr_matrix)

# Row annotations
n_cl = cluster_series_all.max() + 1
pal = plt.get_cmap('tab20', n_cl)
leiden_annot = pd.Series(
    [f'C{cluster_series_all[c] + 1}' for c in sorted_cells_all],
    index=sorted_cells_all, name='Leiden',
)
sample_annot = sample_origin_all[sorted_cells_all].rename('Sample')
leiden_colors = {f'C{i+1}': pal(i % 20) for i in range(n_cl)}

heatmap_kwargs_initial = dict(
    input_type='logr',
    vmin=-2, vmax=2,
    chrom_col='chrom',
    row_cluster=False, col_cluster=False,
    show_chrom_annotation=True, chrom_position='bottom', chrom_labels=True,
    show_row_names=False,
    row_annotation={'Leiden': leiden_annot, 'Sample': sample_annot},
    row_annotation_colors={'Leiden': leiden_colors, 'Sample': SAMPLE_COLORS},
    row_annotation_show_legend=True,
    row_annotation_show_labels=True,
    row_gap=cn.unit(2, 'mm'),
)

title = f'original ({len(cluster_series_all)} cells, LogR)'

cn.cn_heatmap(
    logr_matrix[['chrom', 'start', 'end'] + sorted_cells_all],
    title=title,
    **heatmap_kwargs_initial,
)
plt.show()


In [ ]:
# Chr17-only initial heatmap with ERBB2 span (hg38)
chr17_data = logr_matrix[logr_matrix['chrom'] == 'chr17'].reset_index(drop=True)
chr17_cols = chr17_data[['chrom', 'start', 'end']]

result = cn.cn_heatmap(
    chr17_data[['chrom', 'start', 'end'] + sorted_cells_all],
    title=f'{title} — chr17',
    **{**heatmap_kwargs_initial, 'show_chrom_annotation': False},
)

# ERBB2 ± GENE_SPAN_PADDING (gene range from Ensembl; start/end normalised in case of strand swap)
erbb2 = cn.get_gene_locations(['ERBB2'], cache_file=GENE_CACHE_FILE).iloc[0]
lo = min(erbb2.start, erbb2.end) - GENE_SPAN_PADDING
hi = max(erbb2.start, erbb2.end) + GENE_SPAN_PADDING
hit = chr17_cols[(chr17_cols.end > lo) & (chr17_cols.start < hi)]
if not hit.empty:
    x_lo, x_hi = hit.index[0], hit.index[-1]
    ax = result.axes['heatmap']
    ax.axvspan(x_lo - 0.5, x_hi + 0.5, color='green', alpha=0.30, lw=0)
    ax.axvline(x_lo - 0.5, color='green', alpha=0.40, lw=0.6)
    ax.axvline(x_hi + 0.5, color='green', alpha=0.40, lw=0.6)
    ax.text((x_lo + x_hi) / 2, 0, 'ERBB2', rotation=90, ha='center', va='top',
            transform=ax.get_xaxis_transform(), fontsize=7)
plt.show()


## Remove Normal Cluster & Recluster Tumor

The "normal" cluster has a flat autosomal LogR profile (low std across chr1–22). Identify and drop those cells, then redo UMAP + Leiden + heatmap on tumor cells only. The downstream MAD filter operates on this tumor-only clustering.


In [ ]:
# Identify the "normal" cluster — flat autosomal LogR profile (no CNAs)
autosomes = [f'chr{i}' for i in range(1, 23)]
auto_mask = logr_matrix['chrom'].isin(autosomes)

cluster_std = {}
for lbl in sorted(cluster_series_all.unique()):
    cells = cluster_series_all[cluster_series_all == lbl].index
    median_profile = logr_matrix.loc[auto_mask, cells].median(axis=1)
    cluster_std[lbl] = median_profile.std()

print('Autosomal LogR std (cluster median profile):')
for lbl, s in sorted(cluster_std.items()):
    print(f'  C{lbl+1}: {s:.3f}')

# Auto-flag: any cluster with std < 50% of the next-lowest is "normal"
sorted_stds = sorted(cluster_std.values())
threshold = sorted_stds[1] * 0.5 if len(sorted_stds) >= 2 else 0.0
normal_clusters = [lbl for lbl, s in cluster_std.items() if s < threshold]
print(f'\nNormal cluster(s) (std < {threshold:.3f}): {[f"C{l+1}" for l in normal_clusters]}')

# Drop normal cells
cell_names_tumor = cluster_series_all[~cluster_series_all.isin(normal_clusters)].index.tolist()
print(f'Tumor cells: {len(cell_names_tumor)} kept')

# Re-run UMAP + Leiden on tumor cells
n_neighbors = min(30, len(cell_names_tumor) - 1)
reducer = umap.UMAP(n_components=2, random_state=42,
                    metric='manhattan', min_dist=0.1, n_neighbors=n_neighbors)
umap_coords_tumor = pd.DataFrame(
    reducer.fit_transform(logr_matrix[cell_names_tumor].T.values),
    index=cell_names_tumor, columns=['UMAP1', 'UMAP2'],
)

resolution = 0.05
adata = ad.AnnData(X=logr_matrix[cell_names_tumor].T.values)
adata.obsm['X_umap'] = umap_coords_tumor.values
sc.pp.neighbors(adata, use_rep='X_umap', n_neighbors=n_neighbors)
sc.tl.leiden(adata, resolution=resolution, random_state=42)

leiden_labels_tumor = adata.obs['leiden'].astype(int).values
n_clusters_tumor = leiden_labels_tumor.max() + 1
cluster_series_tumor = pd.Series(leiden_labels_tumor, index=cell_names_tumor, name='cluster')

print(f'\nLeiden after tumor filtering: {n_clusters_tumor} clusters, {len(cell_names_tumor)} cells')
print(cluster_series_tumor.value_counts().sort_index().rename(lambda i: f'C{i+1}'))

# UMAP plots
plot_umap(umap_coords_tumor, cluster_series_tumor.map(lambda i: f'C{i+1}'),
          title=f'UMAP + Leiden ({n_clusters_tumor} clusters) — tumor only')
plot_umap(umap_coords_tumor, sample_origin_all[cell_names_tumor], SAMPLE_COLORS,
          title='UMAP by sample origin — tumor only')

# Tumor heatmap (no MAD filter yet)
sorted_cells_tumor, cluster_link_tumor = compute_heatmap_order(cluster_series_tumor, logr_matrix)
n_cl = cluster_series_tumor.max() + 1
pal = plt.get_cmap('tab20', n_cl)
leiden_annot = pd.Series(
    [f'C{cluster_series_tumor[c] + 1}' for c in sorted_cells_tumor],
    index=sorted_cells_tumor, name='Leiden',
)
sample_annot = sample_origin_all[sorted_cells_tumor].rename('Sample')
leiden_colors = {f'C{i+1}': pal(i % 20) for i in range(n_cl)}

cn.cn_heatmap(
    logr_matrix[['chrom', 'start', 'end'] + sorted_cells_tumor],
    input_type='logr',
    vmin=-2, vmax=2,
    chrom_col='chrom',
    row_cluster=False, col_cluster=False,
    show_chrom_annotation=True, chrom_position='bottom', chrom_labels=True,
    show_row_names=False,
    row_annotation={'Leiden': leiden_annot, 'Sample': sample_annot},
    row_annotation_colors={'Leiden': leiden_colors, 'Sample': SAMPLE_COLORS},
    row_annotation_show_legend=True,
    row_annotation_show_labels=True,
    row_gap=cn.unit(2, 'mm'),
    title=f'tumor only ({len(cluster_series_tumor)} cells, LogR)',
)
plt.show()


## UMAP & Heatmap (After MAD Filtering)

Within each cluster, drop cells whose Manhattan distance to the cluster median LogR profile exceeds `median_dist + k × MAD`. `k=3` is the default used here. Remaining cells are ordered by cluster similarity (complete linkage on cluster medians) and within-cluster Ward linkage for the final heatmap.


In [ ]:
k = 3

# MAD filter
cluster_sized = filter_small_clusters(cluster_series_tumor)
print(f'MAD outlier removal (k={k}):')
cluster_final, _ = filter_mad_outliers(cluster_sized, logr_matrix, k=k)

# Count kept cells per sample
for lbl in SAMPLE_COLORS:
    n = sum(sample_origin_all[c] == lbl for c in cluster_final.index)
    print(f'  → {n} {lbl} kept')

# Order: complete linkage on cluster medians, Ward within clusters
sorted_cells, cluster_link = compute_heatmap_order(cluster_final, logr_matrix)

# UMAP plots after MAD outlier removal
plot_umap(umap_coords_tumor, cluster_final.map(lambda i: f'C{i+1}'),
          title=f'UMAP after MAD outlier removal (k={k})')
plot_umap(umap_coords_tumor, sample_origin_all[cluster_final.index], SAMPLE_COLORS,
          title=f'UMAP after MAD outlier removal (k={k}) — sample origin')

# Row annotations for the heatmap
n_cl = cluster_final.max() + 1
pal = plt.get_cmap('tab20', n_cl)
leiden_annot = pd.Series(
    [f'C{cluster_final[c] + 1}' for c in sorted_cells],
    index=sorted_cells, name='Leiden',
)
sample_annot = sample_origin_all[sorted_cells].rename('Sample')
leiden_colors = {f'C{i+1}': pal(i % 20) for i in range(n_cl)}

heatmap_kwargs = dict(
    input_type='logr',
    vmin=-2, vmax=2,
    chrom_col='chrom',
    row_cluster=False, col_cluster=False,
    show_chrom_annotation=True, chrom_position='bottom', chrom_labels=True,
    show_row_names=False,
    row_annotation={'Leiden': leiden_annot, 'Sample': sample_annot},
    row_annotation_colors={'Leiden': leiden_colors, 'Sample': SAMPLE_COLORS},
    row_annotation_show_legend=True,
    row_cluster_dendrogram=cluster_link,
    row_cluster_dendrogram_size=cn.unit(10, 'mm'),
    row_annotation_show_labels='right',
    row_gap=cn.unit(2, 'mm'),
)

title = f'MAD filtered k={k} ({len(cluster_final)} cells, LogR)'

# Raw filtered heatmap
cn.cn_heatmap(logr_matrix[['chrom', 'start', 'end'] + sorted_cells],
              title=title, **heatmap_kwargs)
plt.show()


In [ ]:
# Chr17-only heatmap with ERBB2 span (hg38)
chr17_data = logr_matrix[logr_matrix['chrom'] == 'chr17'].reset_index(drop=True)
chr17_cols = chr17_data[['chrom', 'start', 'end']]

result = cn.cn_heatmap(
    chr17_data[['chrom', 'start', 'end'] + sorted_cells],
    title=f'{title} — chr17',
    **{**heatmap_kwargs, 'show_chrom_annotation': False},
)

# ERBB2 ± GENE_SPAN_PADDING (gene range from Ensembl; start/end normalised in case of strand swap)
erbb2 = cn.get_gene_locations(['ERBB2'], cache_file=GENE_CACHE_FILE).iloc[0]
lo = min(erbb2.start, erbb2.end) - GENE_SPAN_PADDING
hi = max(erbb2.start, erbb2.end) + GENE_SPAN_PADDING
hit = chr17_cols[(chr17_cols.end > lo) & (chr17_cols.start < hi)]
if not hit.empty:
    x_lo, x_hi = hit.index[0], hit.index[-1]
    ax = result.axes['heatmap']
    ax.axvspan(x_lo - 0.5, x_hi + 0.5, color='green', alpha=0.30, lw=0)
    ax.axvline(x_lo - 0.5, color='green', alpha=0.40, lw=0.6)
    ax.axvline(x_hi + 0.5, color='green', alpha=0.40, lw=0.6)
    ax.text((x_lo + x_hi) / 2, 0, 'ERBB2', rotation=90, ha='center', va='top',
            transform=ax.get_xaxis_transform(), fontsize=7)
plt.show()


## Smoothed Heatmap


In [ ]:
# KNN smoothed (LogR space) — k=15, α=0 matches the knn-smoothing / MAGIC convention
knn_smoothed = knn_smooth(logr_matrix, cluster_final, smooth_k=15, alpha=0)
cn.cn_heatmap(knn_smoothed[['chrom', 'start', 'end'] + sorted_cells],
              title=f'{title} [KNN smoothed k=15, α=0]',
              **heatmap_kwargs)
plt.show()


In [ ]:
# Chr17-only smoothed heatmap with ERBB2 marker
chr17_smoothed = knn_smoothed[knn_smoothed['chrom'] == 'chr17'].reset_index(drop=True)
chr17_cols = chr17_smoothed[['chrom', 'start', 'end']]

result = cn.cn_heatmap(
    chr17_smoothed[['chrom', 'start', 'end'] + sorted_cells],
    title=f'{title} — chr17 [KNN smoothed k=15, α=0]',
    **{**heatmap_kwargs, 'show_chrom_annotation': False},
)

# ERBB2 ± GENE_SPAN_PADDING (gene range from Ensembl; start/end normalised in case of strand swap)
erbb2 = cn.get_gene_locations(['ERBB2'], cache_file=GENE_CACHE_FILE).iloc[0]
lo = min(erbb2.start, erbb2.end) - GENE_SPAN_PADDING
hi = max(erbb2.start, erbb2.end) + GENE_SPAN_PADDING
hit = chr17_cols[(chr17_cols.end > lo) & (chr17_cols.start < hi)]
if not hit.empty:
    x_lo, x_hi = hit.index[0], hit.index[-1]
    ax = result.axes['heatmap']
    ax.axvspan(x_lo - 0.5, x_hi + 0.5, color='green', alpha=0.30, lw=0)
    ax.axvline(x_lo - 0.5, color='green', alpha=0.40, lw=0.6)
    ax.axvline(x_hi + 0.5, color='green', alpha=0.40, lw=0.6)
    ax.text((x_lo + x_hi) / 2, 0, 'ERBB2', rotation=90, ha='center', va='top',
            transform=ax.get_xaxis_transform(), fontsize=7)
plt.show()


## Cluster Median Profiles


### Computation

Compute median LogR profile per cluster using `cluster_final` from the k=3 filtering above.


In [ ]:
unique_lbls = sorted(cluster_final.unique())
n_good = len(unique_lbls)

print(f'Clusters after filtering: {n_good}')
print(cluster_final.value_counts().rename(lambda i: f'C{i+1}').sort_index())

# Compute median LogR per cluster
cluster_cols = {}
for lbl in unique_lbls:
    cells_k = cluster_final[cluster_final == lbl].index
    col_name = f'C{lbl + 1}  (n={len(cells_k)})'
    cluster_cols[col_name] = logr_matrix[cells_k].median(axis=1).values

cluster_logr_matrix = pd.concat(
    [logr_matrix[['chrom', 'start', 'end']],
     pd.DataFrame(cluster_cols, index=logr_matrix.index)],
    axis=1,
)

print(f'\n{n_good} clusters, {len(cluster_final)} total cells')
print(f'Cluster median LogR matrix: {cluster_logr_matrix.shape}')
cluster_logr_matrix.head()


In [ ]:
cluster_labels = [f'C{lbl+1}' for lbl in unique_lbls]
print_tree(cluster_link, cluster_labels)

### Heatmap

One row per Leiden cluster, showing the median LogR profile across all cells in that cluster.


In [ ]:
result_cluster_medians = cn.cn_heatmap(
    cluster_logr_matrix,
    input_type='logr',
    vmin=-2, vmax=2,
    chrom_col='chrom',
    row_cluster=cluster_link,
    col_cluster=False,
    show_chrom_annotation=True,
    chrom_position='bottom',
    chrom_labels=True,
    show_row_names=True,
    show_row_dendrogram=True,
    height=cn.unit(n_good * 12, 'mm'),
    title=f'SC scDNA — {n_good} Cluster Medians (LogR)',
)
plt.show()


### CN Tracks

One stacked CN track per Leiden cluster, using the median CN profile.

In [ ]:
genome_size = genome_range()

# CN color thresholds (CN scale)
cn_thresholds = {'del': 0.5, 'loss': 1.7, 'neutral': 2.3, 'gain': 3.5}
cn_palette = cn.get_cn_palette()

# Derive CN cluster medians locally from LogR cluster medians
cluster_cn_matrix = cn.to_copy_number(cluster_logr_matrix, input_type='logr', ploidy=2)

# Map 'C2' -> full column name 'C2  (n=xxx)'
col_map = {c.split()[0]: c for c in cluster_cn_matrix.columns if c not in ('chrom', 'start', 'end')}

# Order clusters to match the heatmap dendrogram
leaf_order = leaves_list(cluster_link)
ordered_cluster_cols = [list(col_map.values())[i] for i in leaf_order]

# Cluster median tracks, colored by CN value
cluster_tracks = []
for col in ordered_cluster_cols:
    track_df = cluster_cn_matrix[['chrom', 'start', 'end']].copy()
    track_df['value'] = cluster_cn_matrix[col].values
    track_df['_cn_color'] = track_df['value'].apply(
        lambda x: cn.categorize_cn_color(x, thresholds=cn_thresholds)
    )
    cluster_tracks.append(cn.CNPointsTrack(
        track_df,
        y_column='value',
        x_column='start',
        hue_column='_cn_color',
        palette=cn_palette,
        ylabel=col,
        ylim=(0, 9),
        alpha=0.7,
    ))

fig, axes = cn.plot_tracks(cluster_tracks, genome_size, width=8, height_per_track=0.8)
plt.show()
